In [1]:
import os
import pathlib
import geopandas as gpd
from rastervision.core.data import (ClassConfig,
                                    GeoJSONVectorSource,
                                    RasterioSource,
                                    ClassInferenceTransformer,
                                    ObjectDetectionLabelSource,
                                    ObjectDetectionLabels)


2026-06-12 13:06:20:rastervision.pipeline.rv_config: WARNING - Root temporary directory cannot be used: /opt/data/tmp. Using root: /tmp/tmpmedshd4o
/home/VANDERBILT/zimmejr1/anaconda3/envs/DinoV3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
IMAGE_DIR = "/mnt/sarl_commons06/Wernke_projects/GeoPACHA/Imagery_Machine_Learning/Analysis_Images"
# image = "Region01/WV2/506496216060.TIF"
# image_path = os.path.join(IMAGE_DIR,image)
# label_geojson = "./data/Outputs/dinov2_vitl14_upsample_peft_SAA/506496216060_od_labels.geojson"

names_list = ['Corral-Arch','Struct-Unroofed-Arch','Corral-Mod','background']
colors_list = ['blue','green','red','black']
class_config = ClassConfig(
    names=names_list,
    colors=colors_list,
    null_class='background')

In [3]:
def reprocess_labels(aoi_path,label_path,output_path):
        aoi_df = gpd.read_file(aoi_path)
        image  = pathlib.PureWindowsPath(aoi_df['filepath'][0]).as_posix()

        image_path = os.path.join(IMAGE_DIR,image)
        
        raster_source = RasterioSource(image_path,allow_streaming=True)

        vector_source = GeoJSONVectorSource(label_path,crs_transformer = raster_source.crs_transformer,
                                            vector_transformers=[ClassInferenceTransformer(
                    default_class_id=class_config.get_class_id('background'))])

        label_source = ObjectDetectionLabelSource(vector_source)
        labels = label_source.get_labels()
        print(f"Starting with {len(labels)} labels")
        reprocessed_labels = labels.prune_duplicates(labels,score_thresh=.2,merge_thresh=.3)

        reprocessed_labels.save(output_path,class_config=class_config,crs_transformer=raster_source.crs_transformer)



In [ ]:
# label_dir='/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3_stack/data/Outputs/dinov2_vitl14_upsample_peft_SAA'
label_dir='/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3_stack/data/Outputs/dinov2_vitl14_Jun11_2026'
aoi_dir = '/mnt/sarl_commons06/Wernke_projects/zimmejr1/All_AOI_10_20_25'
for label in os.listdir(label_dir):
    if not label.endswith(".geojson"): continue
    image_id = label.split("_")[0]
    label_path = os.path.join(label_dir,label)
    output_path = os.path.join(label_dir,"pruned",f"{image_id}_label_pruned.geojson")
    if os.path.exists(output_path): continue
    print(f"Processing {image_id}")
    aoi_path = os.path.join(aoi_dir,f"imageid_{image_id}.geojson")
    try:
        if os.path.exists(aoi_path):
            reprocess_labels(aoi_path,label_path,output_path)
        else:
            aoi_path = os.path.join(aoi_dir,f"imageid_{image_id}.geojson")
            reprocess_labels(aoi_path,label_path,output_path)
    except:
        print(f"Error with {label_path}")
        continue



Processing 10300100418AF100b
Error with /home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3_stack/data/Outputs/dinov2_vitl14_Jun4_2026/10300100418AF100b_od_labels.geojson
Processing 10300100FC0E2F00
Error with /home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3_stack/data/Outputs/dinov2_vitl14_Jun4_2026/10300100FC0E2F00_od_labels.geojson
Processing 104001002E609200
Starting with 8853 labels


2026-06-12 13:06:36:rastervision.core.data.label_store.object_detection_geojson_store: INFO - Saving 8249 boxes as GeoJSON.


Processing 1030010042129D00a
Error with /home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3_stack/data/Outputs/dinov2_vitl14_Jun4_2026/1030010042129D00a_od_labels.geojson


In [5]:
raster_source = RasterioSource(image_path,allow_streaming=True)

vector_source = GeoJSONVectorSource(label_geojson,crs_transformer = raster_source.crs_transformer,
                                    vector_transformers=[ClassInferenceTransformer(
            default_class_id=class_config.get_class_id('background'))])

label_source = ObjectDetectionLabelSource(vector_source)
labels = label_source.get_labels()

NameError: name 'image_path' is not defined

In [ ]:
reprocess_labels = labels.prune_duplicates(labels,score_thresh=.2,merge_thresh=.3)

In [ ]:
print(len(labels))
print(len(reprocess_labels))

In [ ]:
reprocess_labels.save("506496216060_od_labels_prune.geojson",class_config=class_config,crs_transformer=raster_source.crs_transformer)